In [5]:
from dotenv import load_dotenv
load_dotenv()

True

In [9]:
from dataclasses import dataclass

@dataclass
class Context:
    user_name: str
    is_premium: bool = False

In [11]:
from langchain.agents.middleware import wrap_model_call
from langchain.chat_models import init_chat_model

@wrap_model_call
def dynamic_model_selector(request, handler):
    user_name =request.runtime.context.user_name
    is_premium = request.runtime.context.is_premium
    
    if is_premium:
        model_name = "google_genai:gemini-2.5-flash"
        print(f"{user_name} 사용자는 프리미엄 회원입니다. gemini-2.5-flash를 호출합니다.")
    else:
        model_name = "google_genai:gemini-2.5-flash-lite"
        print(f"{user_name} 사용자는 일반 회원입니다. gemini-2.5-flash-lite를 호출합니다.")

    new_model = init_chat_model(model_name)
    new_request = request.override(model=new_model)

    return handler(new_request)

In [12]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite", 
    tools=[],
    middleware=[dynamic_model_selector],
    context_schema=Context
)

In [13]:
agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요"}]},
    context = Context(user_name="김일남", is_premium=True)
)

김일남 사용자는 프리미엄 회원입니다. gemini-2.5-flash를 호출합니다.


{'messages': [HumanMessage(content='안녕하세요', additional_kwargs={}, response_metadata={}, id='658f0bb9-e796-485b-8d44-19c96a92014f'),
  AIMessage(content='안녕하세요! 반갑습니다.\n\n무엇을 도와드릴까요? 어떤 궁금한 점이 있으신가요?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c0f47-953a-71b0-8cdd-5ef3556bcde0-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 431, 'total_tokens': 433, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 409}})]}

In [14]:
agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요"}]},
    context = Context(user_name="김이남", is_premium=False)
)

김이남 사용자는 일반 회원입니다. gemini-2.5-flash-lite를 호출합니다.


{'messages': [HumanMessage(content='안녕하세요', additional_kwargs={}, response_metadata={}, id='395daeb3-f25d-4423-af10-cc57a31feede'),
  AIMessage(content='안녕하세요! 무엇을 도와드릴까요? 😊', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c0f47-b53f-7f92-84d3-1ca195c4a2a6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 9, 'total_tokens': 11, 'input_token_details': {'cache_read': 0}})]}

---

In [ ]:
from langchain.agents.middleware import wrap_model_call
from langchain.chat_models import init_chat_model

@wrap_model_call
def dynamic_model_selector(request, handler):
    last_msg = request.messages[-1].content if request.messages else ""
    msg_len = len(last_msg)

    print("msg_len:", msg_len)
    
    if msg_len < 10:
        model_name = "google_genai:gemini-2.5-flash-lite"
    elif msg_len < 30:
        model_name = "google_genai:gemini-2.5-flash"
    else:
        model_name = "google_genai:gemini-3-flash-preview"

    new_model = init_chat_model(model_name)
    new_request = request.override(model=new_model)

    return handler(new_request)

In [6]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite", 
    tools=[],
    middleware=[dynamic_model_selector],
)

In [7]:
agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요"}]},
)

{'messages': [HumanMessage(content='안녕하세요', additional_kwargs={}, response_metadata={}, id='d1937ddc-bc98-46b1-924e-982bc9eb999e'),
  AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c0f58-5daf-78b1-8af3-fd60cf0a759a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 8, 'total_tokens': 10, 'input_token_details': {'cache_read': 0}})]}

In [8]:
agent.invoke(
    {"messages": [{"role": "user", "content": "안녕하세요. 저는 김일남입니다. 랭체인에 대해서 궁금한게 많습니다."}]},
)

{'messages': [HumanMessage(content='안녕하세요. 저는 김일남입니다. 랭체인에 대해서 궁금한게 많습니다.', additional_kwargs={}, response_metadata={}, id='bbd156e5-9768-436d-9f53-a844b8f9d1d2'),
  AIMessage(content=[{'type': 'text', 'text': '반갑습니다, 김일남 님! 랭체인(LangChain)에 대해 궁금한 점이 많으시군요. \n\n랭체인은 최근 AI 분야, 특히 **챗GPT와 같은 거대 언어 모델(LLM)을 활용한 서비스를 만들 때 가장 필수적인 도구**로 꼽힙니다. 쉽게 이해하실 수 있도록 핵심 내용을 정리해 드릴게요.\n\n---\n\n### 1. 랭체인(LangChain)이란 무엇인가요?\n한마디로 정의하자면, **"LLM(언어 모델)을 외부 데이터나 도구와 연결하여 강력한 애플리케이션을 만들 수 있게 돕는 프레임워크"**입니다.\n\n비유하자면, 챗GPT가 \'아주 똑똑한 두뇌\'라면, 랭체인은 그 두뇌가 \'책을 읽고(문서 참조)\', \'계산기를 쓰고(도구 활용)\', \'과거를 기억하며(메모리)\', \'복잡한 일을 순서대로 처리(체인)\'할 수 있도록 팔다리와 도구를 붙여주는 역할을 합니다.\n\n### 2. 왜 랭체인을 쓰나요? (주요 특징)\n그냥 챗GPT API만 써도 되는데 왜 랭체인이 필요할까요?\n*   **데이터 연결 (Retrieval):** 모델이 학습하지 않은 최신 정보나 우리 회사 내부 문서(PDF, 엑셀 등)를 읽어서 답변하게 할 수 있습니다. (RAG 기술의 핵심)\n*   **체이닝 (Chains):** "질문을 요약하고 -> 답변을 생성하고 -> 답변을 번역해라" 같은 여러 단계의 작업을 하나로 묶어 자동화합니다.\n*   **기억력 (Memory):** 이전 대화 내용을 기억하게 하여 맥락 있는 대화를 이어가게 합니다.\n*   **에이전트 (Agents):** AI가 스스로 판단해서